# CICIDS2017 TensorFlow MLP temporal-shift experiment

This Kaggle-ready notebook evaluates the `MLP baseline` for binary Network Intrusion Detection on CICIDS-2017. It mirrors the PyTorch EVT experiment so framework behavior can be compared under the same temporal-shift and reproducibility protocol.


## 2. Install/import dependencies

Kaggle notebooks include the core scientific Python stack. The install cell is intentionally minimal and can be extended if your runtime is missing a package.

In [ ]:
# Kaggle usually has these installed. Uncomment if your runtime is missing packages.
# !pip install -q pandas numpy scikit-learn tensorflow

In [ ]:
import os
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISM"] = "1"

import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import genpareto, kstest
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))



In [ ]:
DATASET_NAME = "CICIDS-2017"
FRAMEWORK = "TensorFlow/Keras"
NOTEBOOK_FILENAME = "CICIDS2017_TensorFlow_MLP.ipynb"
MODEL_FAMILY = "MLP"
TASK_NAME = "Binary NIDS temporal-shift EVT reproducibility"



## 3. Set reproducibility configuration

Each TensorFlow baseline runs `10 training seeds x 3 weight initialization tuples = 30 runs` per threshold strategy. Report model quality with aggregate statistics rather than a single run.


In [ ]:
TRAINING_SEEDS = [57, 305, 5, 9667, 405, 111, 222, 333, 444, 555]
WEIGHT_INIT_SEEDS = {
    "W1": [1004, 77, 259, 35],
    "W2": [8, 358, 200, 35],
    "W3": [487, 22, 900, 7],
}

DEFAULT_SEED = TRAINING_SEEDS[0]

def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_global_seed(DEFAULT_SEED)
print(f"Configured {len(TRAINING_SEEDS) * len(WEIGHT_INIT_SEEDS)} runs per threshold strategy.")



## 4. Create data and Kaggle results folders\n\nGenerated CSV results are written to `/kaggle/working/results/`.\n

In [ ]:
DATA_DIR = Path("/kaggle/working/data")
RESULTS_DIR = Path("/kaggle/working/results")
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Final dataset path:", DATA_DIR)
print("Results directory:", RESULTS_DIR)

In [ ]:
RESULTS_CSV_PATH = RESULTS_DIR / "CICIDS2017_TensorFlow_MLP_results.csv"
AGGREGATED_RESULTS_CSV_PATH = RESULTS_DIR / "CICIDS2017_TensorFlow_MLP_aggregated_results.csv"

print("Per-run results CSV:", RESULTS_CSV_PATH)
print("Aggregated results CSV:", AGGREGATED_RESULTS_CSV_PATH)


## 5. Prepare dataset from Kaggle Add Input

Before running this notebook on Kaggle, attach the CICIDS-2017 dataset with **Add Input**. The notebook reads files from `/kaggle/input`, copies the weekday CSV files into `/kaggle/working/data`, and then runs the same preprocessing path for both TensorFlow notebooks.


In [ ]:
# Dataset download is not performed in non-interactive Kaggle runs.
# Use Kaggle Add Input to attach the CICIDS-2017 dataset before running all cells.



In [ ]:
import shutil

KAGGLE_INPUT_DIR = Path("/kaggle/input")
CICIDS_WEEKDAY_HINTS = ("monday", "tuesday", "wednesday", "thursday", "friday")


def _looks_like_cicids_csv(path: Path) -> bool:
    name = path.name.lower()
    return path.suffix.lower() == ".csv" and "_plus" not in name and any(day in name for day in CICIDS_WEEKDAY_HINTS)


def copy_attached_cicids_files(input_root: Path, target_root: Path) -> list[Path]:
    if not input_root.exists():
        return []

    source_files = sorted(path for path in input_root.rglob("*.csv") if _looks_like_cicids_csv(path))
    if not source_files:
        return []

    copied_files = []
    for source_file in source_files:
        relative_path = source_file.relative_to(input_root)
        target_path = target_root / relative_path
        target_path.parent.mkdir(parents=True, exist_ok=True)
        if not target_path.exists() or source_file.stat().st_size != target_path.stat().st_size:
            shutil.copy2(source_file, target_path)
        copied_files.append(target_path)
    return copied_files


DATA_DIR.mkdir(parents=True, exist_ok=True)
copied_files = copy_attached_cicids_files(KAGGLE_INPUT_DIR, DATA_DIR)

print("Kaggle input directory:", KAGGLE_INPUT_DIR)
print(f"CICIDS-2017 weekday CSV files copied: {len(copied_files)}")
for csv_file in copied_files:
    print(" -", csv_file)

csv_files = sorted(DATA_DIR.rglob("*.csv"))
print("Final dataset path:", DATA_DIR)
print(f"CSV files available in working directory: {len(csv_files)}")
for csv_file in csv_files:
    print(" -", csv_file)

if not csv_files:
    available_inputs = sorted(str(path) for path in KAGGLE_INPUT_DIR.glob("*") if path.exists()) if KAGGLE_INPUT_DIR.exists() else []
    raise FileNotFoundError(
        "No CICIDS-2017 weekday CSV files were found in /kaggle/input. "
        "On Kaggle, click Add Input and attach the dataset that contains Monday-Friday CICIDS-2017 CSV files. "
        f"Visible input roots: {available_inputs}"
    )



## 6. Load dataset from /kaggle/working/data

This loader recursively finds CSV files with `Path("/kaggle/working/data").rglob("*.csv")`, adds a `source_file` column before concatenation, prints the discovered files, reports dataset shape, and prints label distribution.

In [ ]:
LABEL_CANDIDATES = ["Label", "label", "Class", "class", "Attack", "attack"]
CICIDS_ORIGINAL_WEEKDAYS = ("monday", "tuesday", "wednesday", "thursday", "friday")


def detect_label_column(df: pd.DataFrame) -> str:
    for candidate in LABEL_CANDIDATES:
        if candidate in df.columns:
            return candidate

    lower_map = {col.lower(): col for col in df.columns}
    for candidate in LABEL_CANDIDATES:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    object_columns = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    object_columns = [col for col in object_columns if col != "source_file"]
    if object_columns:
        return object_columns[-1]

    raise ValueError(
        "Could not automatically detect a label column. Rename the target column to one of: "
        + ", ".join(LABEL_CANDIDATES)
    )

def load_csv_dataset(data_dir: Path) -> pd.DataFrame:
    csv_files = sorted(data_dir.rglob("*.csv"))
    csv_files = [
        csv_path
        for csv_path in csv_files
        if "_plus" not in csv_path.name.lower()
        and any(day in csv_path.name.lower() for day in CICIDS_ORIGINAL_WEEKDAYS)
    ]
    print(f"CSV files found after excluding *_plus.csv files: {len(csv_files)}")
    print("CSV files used:")
    for csv_path in csv_files:
        print(" -", csv_path.relative_to(data_dir))

    if not csv_files:
        raise FileNotFoundError(
            "No CICIDS-2017 weekday CSV files found under /kaggle/working/data. "
            "Attach a Kaggle input dataset containing Monday-Friday CICIDS-2017 CSV files."
        )

    frames = []
    for csv_path in csv_files:
        print(f"Loading {csv_path}")
        frame = pd.read_csv(csv_path, low_memory=False)
        frame.columns = frame.columns.astype(str).str.strip()
        frame["source_file"] = str(csv_path.relative_to(data_dir))
        frames.append(frame)

    df = pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]
    df.columns = df.columns.astype(str).str.strip()

    print(f"Dataset shape: {df.shape}")
    print("Column names:")
    print(list(df.columns))

    label_col = detect_label_column(df)
    print(f"Detected label column: {label_col}")
    print("Label distribution:")
    print(df[label_col].astype(str).str.strip().value_counts(dropna=False))
    return df

raw_df = load_csv_dataset(DATA_DIR)
raw_df.head()


## 7. Data preprocessing

Preprocessing is fit after the paper-style train/test protocol is built. The label encoder is fit on training labels when possible, and `StandardScaler` is fit only on `X_train` before transforming `X_test`.

In [ ]:
def clean_label_series(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip()

def is_benign_label(label: str) -> bool:
    normalized = str(label).strip().lower()
    return normalized in {"benign", "normal", "normal traffic", "background"} or "benign" in normalized

def to_binary_attack_labels(labels: pd.Series) -> pd.Series:
    return clean_label_series(labels).apply(lambda value: "Benign" if is_benign_label(value) else "Attack")

def encode_binary_attack_labels(labels: pd.Series) -> np.ndarray:
    return to_binary_attack_labels(labels).map({"Benign": 0, "Attack": 1}).astype(np.int64).to_numpy()

def make_binary_label_encoder() -> LabelEncoder:
    encoder = LabelEncoder()
    encoder.classes_ = np.array(["Benign", "Attack"])
    return encoder

def _numeric_feature_frame(df: pd.DataFrame, label_col: str, numeric_cols: list[str] | None = None) -> pd.DataFrame:
    feature_df = df.drop(columns=[label_col], errors="ignore").copy()
    feature_df.drop(columns=["source_file"], inplace=True, errors="ignore")
    for col in feature_df.columns:
        if feature_df[col].dtype == "object":
            feature_df[col] = pd.to_numeric(feature_df[col].astype(str).str.strip(), errors="coerce")
    if numeric_cols is None:
        numeric_cols = feature_df.select_dtypes(include=[np.number]).columns.tolist()
        if not numeric_cols:
            raise ValueError("No numeric feature columns were found after protocol splitting.")
    return feature_df.reindex(columns=numeric_cols).replace([np.inf, -np.inf], np.nan)

def prepare_feature_matrices(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    calibration_df: pd.DataFrame | None,
    label_col: str,
):
    X_train_df = _numeric_feature_frame(train_df, label_col)
    numeric_cols = X_train_df.columns.tolist()
    X_test_df = _numeric_feature_frame(test_df, label_col, numeric_cols)
    X_calibration_df = _numeric_feature_frame(calibration_df, label_col, numeric_cols) if calibration_df is not None else None

    train_medians = X_train_df.median(numeric_only=True).fillna(0)
    X_train_df = X_train_df.fillna(train_medians).fillna(0)
    X_test_df = X_test_df.fillna(train_medians).fillna(0)
    if X_calibration_df is not None:
        X_calibration_df = X_calibration_df.fillna(train_medians).fillna(0)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_df).astype(np.float32)
    X_test = scaler.transform(X_test_df).astype(np.float32)
    X_calibration = scaler.transform(X_calibration_df).astype(np.float32) if X_calibration_df is not None else None
    return X_train, X_test, X_calibration, scaler, numeric_cols

def fit_label_encoder_train_when_possible(y_train_raw: pd.Series, y_test_raw: pd.Series, y_calibration_raw: pd.Series | None = None):
    y_train_raw = clean_label_series(y_train_raw)
    y_test_raw = clean_label_series(y_test_raw)
    extra = [] if y_calibration_raw is None else [clean_label_series(y_calibration_raw)]
    observed_test = pd.concat([y_test_raw, *extra], ignore_index=True) if extra else y_test_raw
    train_classes = set(y_train_raw.unique())
    test_classes = set(observed_test.unique())
    unseen_test_classes = sorted(test_classes - train_classes)

    label_encoder = LabelEncoder()
    if unseen_test_classes:
        print("Shift labels not present in training labels; fitting LabelEncoder on train+shift labels for transform compatibility:")
        print(unseen_test_classes)
        label_encoder.fit(pd.concat([y_train_raw, observed_test], ignore_index=True))
    else:
        label_encoder.fit(y_train_raw)

    y_calibration = None if y_calibration_raw is None else label_encoder.transform(clean_label_series(y_calibration_raw))
    return label_encoder, label_encoder.transform(y_train_raw), label_encoder.transform(y_test_raw), y_calibration

def prepare_protocol_data(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    *,
    calibration_df: pd.DataFrame | None = None,
    binary_labels: bool = True,
    metadata: dict | None = None,
):
    global X_train, X_test, X_threshold_calibration, y_train, y_test, y_threshold_calibration
    global label_encoder, scaler, feature_columns
    global NUM_FEATURES, NUM_CLASSES, IS_BINARY, RUN_METADATA

    label_probe = [train_df.head(1), test_df.head(1)]
    if calibration_df is not None and not calibration_df.empty:
        label_probe.append(calibration_df.head(1))
    label_col = detect_label_column(pd.concat(label_probe, ignore_index=True))

    train_df = train_df.copy().replace([np.inf, -np.inf], np.nan).dropna(subset=[label_col])
    test_df = test_df.copy().replace([np.inf, -np.inf], np.nan).dropna(subset=[label_col])
    calibration_df = None if calibration_df is None else calibration_df.copy().replace([np.inf, -np.inf], np.nan).dropna(subset=[label_col])
    if calibration_df is not None and calibration_df.empty:
        calibration_df = None

    X_train, X_test, X_threshold_calibration, scaler, feature_columns = prepare_feature_matrices(
        train_df, test_df, calibration_df, label_col
    )

    if binary_labels:
        label_encoder = make_binary_label_encoder()
        y_train = encode_binary_attack_labels(train_df[label_col])
        y_test = encode_binary_attack_labels(test_df[label_col])
        y_threshold_calibration = None if calibration_df is None else encode_binary_attack_labels(calibration_df[label_col])
    else:
        y_calibration_raw = None if calibration_df is None else calibration_df[label_col]
        label_encoder, y_train, y_test, y_threshold_calibration = fit_label_encoder_train_when_possible(
            train_df[label_col], test_df[label_col], y_calibration_raw
        )
        y_train = y_train.astype(np.int64)
        y_test = y_test.astype(np.int64)
        if y_threshold_calibration is not None:
            y_threshold_calibration = y_threshold_calibration.astype(np.int64)

    NUM_FEATURES = X_train.shape[1]
    NUM_CLASSES = len(label_encoder.classes_)
    IS_BINARY = NUM_CLASSES == 2
    RUN_METADATA = metadata.copy() if metadata else {}

    print("Train shape:", X_train.shape, y_train.shape)
    print("Threshold calibration shape:", None if X_threshold_calibration is None else X_threshold_calibration.shape)
    print("Shifted test shape:", X_test.shape, y_test.shape)
    print(f"Classes ({NUM_CLASSES}): {list(label_encoder.classes_)}")
    print("Scaler fitted on training data only.")
    return X_train, X_test, y_train, y_test




## 8. Temporal-shift train/calibration/test split

CICIDS-2017 is split chronologically: train on the first two days/files, reserve 10% labeled samples from the first shifted day for EVT calibration, then evaluate on the remaining shifted traffic from day 3 onward.


In [ ]:
import re

CALIBRATION_FRACTION = 0.10
CALIBRATION_RANDOM_STATE = 2026

WEEKDAY_ORDER = {
    "monday": 0,
    "tuesday": 1,
    "wednesday": 2,
    "thursday": 3,
    "friday": 4,
    "saturday": 5,
    "sunday": 6,
}

def _source_file_sort_key(source_file: str):
    name = Path(source_file).name.lower()
    for weekday, rank in WEEKDAY_ORDER.items():
        if weekday in name:
            return (0, rank, name)

    date_match = re.search(r"(20\d{2})[-_ ]?(\d{1,2})[-_ ]?(\d{1,2})", name)
    if date_match:
        year, month, day = map(int, date_match.groups())
        return (1, year, month, day, name)

    number_match = re.search(r"(\d+)", name)
    if number_match:
        return (2, int(number_match.group(1)), name)

    return None

def build_cicids2017_temporal_split(df: pd.DataFrame):
    if "source_file" not in df.columns:
        raise ValueError("CICIDS-2017 temporal split requires a source_file column added during CSV loading.")

    source_files = sorted(df["source_file"].dropna().astype(str).unique())
    sort_keys = {source_file: _source_file_sort_key(source_file) for source_file in source_files}
    failed = [source_file for source_file, key in sort_keys.items() if key is None]
    if failed:
        raise ValueError(
            "Could not detect chronological day/order from source_file names. "
            f"Available source_file names: {source_files}"
        )

    ordered_files = sorted(source_files, key=lambda source_file: sort_keys[source_file])

    def group_key(source_file: str):
        key = sort_keys[source_file]
        return key[:-1]

    ordered_day_keys = []
    for source_file in ordered_files:
        key = group_key(source_file)
        if key not in ordered_day_keys:
            ordered_day_keys.append(key)

    if len(ordered_day_keys) < 3:
        raise ValueError(f"Need at least 3 source files/days for CICIDS-2017 protocol. Found: {ordered_files}")

    train_day_keys = set(ordered_day_keys[:2])
    first_shift_day_key = ordered_day_keys[2]
    train_files = [source_file for source_file in ordered_files if group_key(source_file) in train_day_keys]
    shift_files = [source_file for source_file in ordered_files if group_key(source_file) not in train_day_keys]
    calibration_files = [source_file for source_file in shift_files if group_key(source_file) == first_shift_day_key]

    train_df = df[df["source_file"].isin(train_files)].copy()
    shifted_df = df[df["source_file"].isin(shift_files)].copy()
    first_shift_df = df[df["source_file"].isin(calibration_files)].copy()
    label_col = detect_label_column(first_shift_df)

    calibration_df = first_shift_df.groupby(label_col, group_keys=False).apply(
        lambda group: group.sample(
            n=max(1, int(round(len(group) * CALIBRATION_FRACTION))),
            random_state=CALIBRATION_RANDOM_STATE,
        )
    )
    calibration_indices = set(calibration_df.index)
    test_df = shifted_df.loc[~shifted_df.index.isin(calibration_indices)].copy()

    print("CICIDS-2017 temporal-shift protocol")
    print("Train: first two chronological days/files")
    for file_name in train_files:
        print(" -", file_name)
    print("Threshold calibration: 10% labeled samples from the first shifted day")
    for file_name in calibration_files:
        print(" -", file_name)
    print("Shifted test: remaining samples from day 3 onward")
    for file_name in shift_files:
        print(" -", file_name)
    print(f"Calibration rows: {len(calibration_df):,}")
    print(f"Shifted test rows after calibration holdout: {len(test_df):,}")

    return train_df, calibration_df.copy(), test_df, train_files, calibration_files, shift_files

train_df, calibration_df, test_df, train_files, calibration_files, test_files = build_cicids2017_temporal_split(raw_df)
prepare_protocol_data(
    train_df,
    test_df,
    calibration_df=calibration_df,
    binary_labels=True,
    metadata={
        "task": "binary_intrusion_detection",
        "temporal_protocol": "train_day_1_2_test_day_3_5",
        "calibration_fraction": CALIBRATION_FRACTION,
        "train_files": json.dumps(train_files),
        "calibration_files": json.dumps(calibration_files),
        "test_files": json.dumps(test_files),
    },
)




## 9. Define evaluation metrics

Metrics use scikit-learn. Multi-class precision, recall, and F1 use `average="weighted"`; binary experiments use binary averaging.

In [ ]:
def metric_average(num_classes: int) -> str:
    return "binary" if num_classes == 2 else "weighted"

def false_positive_rate(y_true, y_pred) -> float:
    if NUM_CLASSES != 2:
        return float("nan")
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return float(fp / (fp + tn)) if (fp + tn) else 0.0

def evaluate_predictions(y_true, y_pred, y_score=None, labels=None) -> dict:
    average = metric_average(NUM_CLASSES)
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average=average, zero_division=0),
        "recall": recall_score(y_true, y_pred, average=average, zero_division=0),
        "f1": f1_score(y_true, y_pred, average=average, zero_division=0),
        "fpr": false_positive_rate(y_true, y_pred),
        "roc_auc": float("nan"),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES))).tolist(),
    }
    if y_score is not None and NUM_CLASSES == 2 and len(np.unique(y_true)) == 2:
        try:
            metrics["roc_auc"] = roc_auc_score(y_true, y_score)
        except Exception:
            metrics["roc_auc"] = float("nan")
    try:
        metrics["classification_report"] = classification_report(
            y_true,
            y_pred,
            target_names=labels,
            zero_division=0,
            output_dict=True,
        )
    except Exception:
        metrics["classification_report"] = classification_report(
            y_true,
            y_pred,
            zero_division=0,
            output_dict=True,
        )
    return metrics

def _aggregate_with_group_cols(per_run_df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    metric_cols = ["accuracy", "precision", "recall", "f1", "fpr", "roc_auc"]
    aggregated = per_run_df.groupby(group_cols, dropna=False)[metric_cols].agg(["mean", "std", "min", "max"]).round(6)
    aggregated.columns = [f"{metric}_{stat}" for metric, stat in aggregated.columns]
    return aggregated.reset_index()

def aggregate_results(per_run_df: pd.DataFrame) -> pd.DataFrame:
    base_cols = ["dataset_name", "framework", "model_name", "task", "temporal_protocol"]
    base_cols = [col for col in base_cols if col in per_run_df.columns]
    return _aggregate_with_group_cols(per_run_df, base_cols)




## 10. Training, reproducibility, and EVT threshold utilities

Every run uses one training seed and one named weight-initialization tuple. The improved thresholding strategy uses adaptive EVT threshold estimation with top-10% tail selection, GPD fitting, KS-based contamination pruning, and a target false alarm probability alpha0=0.05.


In [ ]:
EVT_TAIL_FRACTION = 0.10
EVT_TARGET_FALSE_ALARM_PROBABILITY = 0.05
EVT_MIN_TAIL_SAMPLES = 5
PLOTS_DIR = RESULTS_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


class EVTThresholdEstimator:
    """Adaptive EVT threshold estimator for TensorFlow anomaly scores."""

    def __init__(
        self,
        validation_scores,
        tail_fraction: float = EVT_TAIL_FRACTION,
        alpha0: float = EVT_TARGET_FALSE_ALARM_PROBABILITY,
        min_tail_samples: int = EVT_MIN_TAIL_SAMPLES,
    ):
        self.validation_scores = np.asarray(validation_scores, dtype=float)
        self.validation_scores = self.validation_scores[np.isfinite(self.validation_scores)]
        if len(self.validation_scores) < min_tail_samples:
            raise ValueError("EVT threshold estimation needs more finite validation scores.")
        self.tail_fraction = tail_fraction
        self.alpha0 = alpha0
        self.min_tail_samples = min_tail_samples
        self.n = int(len(self.validation_scores))
        self.u = None
        self.initial_excesses = None
        self.remaining_excesses = None
        self.removed_excesses = None
        self.ks_history = []
        self.xi_hat = None
        self.sigma_hat = None
        self.nu = None
        self.threshold = None

    @staticmethod
    def _fit_gpd(excesses):
        values = np.asarray(excesses, dtype=float)
        values = values[np.isfinite(values)]
        if len(values) < 2 or np.allclose(values, values[0]):
            return 0.0, max(float(values.std(ddof=0)), 1e-12)
        xi, loc, sigma = genpareto.fit(values, floc=0)
        return float(xi), max(float(sigma), 1e-12)

    @staticmethod
    def _ks_statistic(excesses, xi, sigma):
        values = np.asarray(excesses, dtype=float)
        if len(values) < 2:
            return float("inf")
        return float(kstest(values, "genpareto", args=(xi, 0, sigma)).statistic)

    def _select_tail_excesses(self):
        sorted_scores = np.sort(self.validation_scores)
        tail_count = max(self.min_tail_samples, int(math.ceil(len(sorted_scores) * self.tail_fraction)))
        tail_count = min(tail_count, len(sorted_scores) - 1)
        self.u = float(sorted_scores[-tail_count])
        excesses = sorted_scores[sorted_scores > self.u] - self.u
        if len(excesses) < self.min_tail_samples:
            excesses = sorted_scores[-tail_count:] - self.u
            excesses = excesses[excesses >= 0]
        self.initial_excesses = np.sort(excesses.astype(float))
        if len(self.initial_excesses) < self.min_tail_samples:
            raise ValueError("Not enough EVT tail excesses after high-threshold selection.")

    def _find_best_pruning(self):
        sorted_excesses = np.sort(self.initial_excesses)
        max_removed = max(0, len(sorted_excesses) - self.min_tail_samples)
        best = None
        self.ks_history = []
        for removed_count in range(max_removed + 1):
            remaining = sorted_excesses[: len(sorted_excesses) - removed_count]
            xi, sigma = self._fit_gpd(remaining)
            ks_stat = self._ks_statistic(remaining, xi, sigma)
            row = {
                "removed_count": removed_count,
                "ks_statistic": ks_stat,
                "xi": xi,
                "sigma": sigma,
                "remaining_tail_samples": len(remaining),
            }
            self.ks_history.append(row)
            if best is None or ks_stat < best["ks_statistic"]:
                best = row
        best_removed = int(best["removed_count"])
        self.remaining_excesses = sorted_excesses[: len(sorted_excesses) - best_removed]
        self.removed_excesses = sorted_excesses[len(sorted_excesses) - best_removed :] if best_removed else np.array([])
        self.xi_hat = float(best["xi"])
        self.sigma_hat = float(best["sigma"])
        self.nu = int(best["remaining_tail_samples"])

    def _compute_threshold(self):
        ratio = self.n / max(self.nu * self.alpha0, 1e-12)
        if abs(self.xi_hat) < 1e-8:
            self.threshold = self.u + self.sigma_hat * math.log(ratio)
        else:
            self.threshold = self.u + (self.sigma_hat / self.xi_hat) * (math.pow(ratio, self.xi_hat) - 1.0)
        return float(self.threshold)

    def fit(self):
        self._select_tail_excesses()
        self._find_best_pruning()
        self._compute_threshold()
        return self

    def predict(self, scores) -> np.ndarray:
        if self.threshold is None:
            raise RuntimeError("Call fit() before predict().")
        scores = np.asarray(scores, dtype=float)
        return (scores > self.threshold).astype(np.int64)

    def summary(self) -> dict:
        return {
            "evt_u": self.u,
            "evt_xi_hat": self.xi_hat,
            "evt_sigma_hat": self.sigma_hat,
            "evt_n": self.n,
            "evt_nu": self.nu,
            "evt_alpha0": self.alpha0,
            "evt_tail_fraction": self.tail_fraction,
            "evt_removed_outliers": 0 if self.removed_excesses is None else int(len(self.removed_excesses)),
            "evt_initial_tail_samples": 0 if self.initial_excesses is None else int(len(self.initial_excesses)),
            "evt_threshold": self.threshold,
            "evt_min_ks_statistic": min((row["ks_statistic"] for row in self.ks_history), default=float("nan")),
        }


def initializer_from_seed(seed_value: int):
    return keras.initializers.GlorotUniform(seed=seed_value)

def weight_seed(seed_tuple, index: int) -> int:
    return int(seed_tuple[index % len(seed_tuple)])

def compile_model(model, lr: float = 1e-3):
    loss = keras.losses.BinaryCrossentropy(from_logits=True) if IS_BINARY else keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss=loss)
    return model

def predict_tf(model, X_values: np.ndarray, threshold: float = 0.5):
    logits = model.predict(X_values, verbose=0)
    if IS_BINARY:
        probabilities = tf.sigmoid(logits).numpy().reshape(-1)
        return (probabilities >= threshold).astype(int), probabilities
    probabilities = tf.nn.softmax(logits, axis=1).numpy()
    return probabilities.argmax(axis=1), probabilities

def validation_scores_for_evt(model) -> np.ndarray:
    if X_threshold_calibration is None or y_threshold_calibration is None or len(y_threshold_calibration) == 0:
        _, train_scores = predict_tf(model, X_train)
        return train_scores
    _, calibration_scores = predict_tf(model, X_threshold_calibration)
    return calibration_scores

def predict_with_evt(model):
    if not IS_BINARY:
        raise ValueError("EVT thresholding is implemented for binary anomaly scores only.")
    validation_scores = validation_scores_for_evt(model)
    _, test_scores = predict_tf(model, X_test)
    estimator = EVTThresholdEstimator(
        validation_scores,
        tail_fraction=EVT_TAIL_FRACTION,
        alpha0=EVT_TARGET_FALSE_ALARM_PROBABILITY,
        min_tail_samples=EVT_MIN_TAIL_SAMPLES,
    ).fit()
    y_pred = estimator.predict(test_scores)
    return y_pred, test_scores, estimator.summary(), estimator, validation_scores

def build_class_weight_dict():
    present_classes = np.unique(y_train)
    weights = np.ones(NUM_CLASSES, dtype=np.float32)
    computed_weights = compute_class_weight(class_weight="balanced", classes=present_classes, y=y_train)
    weights[present_classes] = computed_weights
    return {int(cls): float(weights[cls]) for cls in range(NUM_CLASSES)}

def plot_confusion_matrix_array(y_true, y_pred, title: str, output_path: Path):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4.5, 4))
    image = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks([0, 1], labels=["Benign", "Attack"])
    ax.set_yticks([0, 1], labels=["Benign", "Attack"])
    for row in range(cm.shape[0]):
        for col in range(cm.shape[1]):
            ax.text(col, row, int(cm[row, col]), ha="center", va="center")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.close(fig)

def plot_evt_diagnostics(validation_scores, test_scores, estimator: EVTThresholdEstimator, output_prefix: Path):
    validation_scores = np.asarray(validation_scores, dtype=float)
    test_scores = np.asarray(test_scores, dtype=float)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(validation_scores, bins=50, alpha=0.65, label="Validation-shift scores")
    ax.hist(test_scores, bins=50, alpha=0.35, label="Test scores")
    ax.axvline(estimator.u, color="orange", linestyle="--", label="EVT high threshold u")
    ax.axvline(estimator.threshold, color="red", linestyle="--", label="EVT final threshold")
    ax.set_title("Anomaly score distribution")
    ax.set_xlabel("Anomaly score")
    ax.set_ylabel("Count")
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_prefix.with_name(output_prefix.name + "_score_distribution.png"), dpi=160)
    plt.close(fig)

    if estimator.remaining_excesses is not None and len(estimator.remaining_excesses) > 0:
        sorted_excesses = np.sort(estimator.remaining_excesses)
        empirical_cdf = np.arange(1, len(sorted_excesses) + 1) / len(sorted_excesses)
        fitted_cdf = genpareto.cdf(sorted_excesses, estimator.xi_hat, loc=0, scale=estimator.sigma_hat)
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.plot(sorted_excesses, empirical_cdf, label="Empirical tail CDF")
        ax.plot(sorted_excesses, fitted_cdf, label="Fitted GPD CDF")
        ax.set_title("GPD fitting on pruned tail excesses")
        ax.set_xlabel("Tail excess")
        ax.set_ylabel("CDF")
        ax.legend()
        fig.tight_layout()
        fig.savefig(output_prefix.with_name(output_prefix.name + "_gpd_fit.png"), dpi=160)
        plt.close(fig)

    if estimator.ks_history:
        history = pd.DataFrame(estimator.ks_history)
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.plot(history["removed_count"], history["ks_statistic"], marker="o")
        ax.set_title("KS statistic vs removed tail outliers")
        ax.set_xlabel("Removed largest tail samples")
        ax.set_ylabel("KS statistic")
        fig.tight_layout()
        fig.savefig(output_prefix.with_name(output_prefix.name + "_ks_vs_removed.png"), dpi=160)
        plt.close(fig)

def should_plot_diagnostics(training_seed: int, weight_init_name: str) -> bool:
    return training_seed == DEFAULT_SEED and weight_init_name == "W1"

def safe_name(value: str) -> str:
    return str(value).replace(" ", "_").replace("/", "_").replace(".", "_")

def train_tf_model(
    model_builder,
    model_name: str,
    training_seed: int,
    weight_init_name: str,
    weight_init_tuple,
    epochs: int = 5,
    batch_size: int = 512,
    lr: float = 1e-3,
    use_class_weights: bool = True,
    threshold_strategy: str = "default_0.5",
):
    set_global_seed(training_seed)
    tf.keras.backend.clear_session()
    model = compile_model(model_builder(weight_init_tuple), lr=lr)
    class_weight = build_class_weight_dict() if use_class_weights else None
    model.fit(
        X_train,
        y_train.astype(np.float32 if IS_BINARY else np.int64),
        epochs=epochs,
        batch_size=batch_size,
        verbose=0,
        shuffle=True,
        class_weight=class_weight,
    )

    evt_summary = {}
    if threshold_strategy == "default_0.5":
        threshold = 0.5
        y_pred, test_scores = predict_tf(model, X_test, threshold=threshold)
    elif threshold_strategy == "evt_adaptive_threshold":
        y_pred, test_scores, evt_summary, estimator, validation_scores = predict_with_evt(model)
        threshold = evt_summary["evt_threshold"]
        if should_plot_diagnostics(training_seed, weight_init_name):
            prefix = PLOTS_DIR / f"{safe_name(model_name)}_{threshold_strategy}_seed{training_seed}_{weight_init_name}"
            plot_evt_diagnostics(validation_scores, test_scores, estimator, prefix)
    else:
        raise ValueError(f"Unknown threshold strategy: {threshold_strategy}")

    if should_plot_diagnostics(training_seed, weight_init_name):
        cm_path = PLOTS_DIR / f"{safe_name(model_name)}_{threshold_strategy}_confusion_matrix.png"
        plot_confusion_matrix_array(y_test, y_pred, f"{model_name} - {threshold_strategy}", cm_path)

    metrics = evaluate_predictions(y_test, y_pred, y_score=test_scores, labels=list(label_encoder.classes_))
    return {
        "dataset_name": DATASET_NAME,
        "framework": FRAMEWORK,
        "model_name": model_name,
        "threshold_strategy": threshold_strategy,
        "training_seed": training_seed,
        "weight_init_name": weight_init_name,
        "weight_init_tuple": json.dumps(weight_init_tuple),
        **RUN_METADATA,
        "threshold": threshold,
        **evt_summary,
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "fpr": metrics["fpr"],
        "roc_auc": metrics["roc_auc"],
        "confusion_matrix": metrics["confusion_matrix"],
        "classification_report": metrics["classification_report"],
    }


import gc

def append_result_row(row: dict, csv_path: Path) -> None:
    row_to_save = row.copy()
    for col in ["confusion_matrix", "classification_report"]:
        if col in row_to_save:
            row_to_save[col] = json.dumps(row_to_save[col])
    pd.DataFrame([row_to_save]).to_csv(
        csv_path,
        mode="a",
        index=False,
        header=not csv_path.exists(),
    )

def cleanup_after_run() -> None:
    gc.collect()
    tf.keras.backend.clear_session()

def run_model_experiment(model_builder, model_name: str, threshold_strategy: str, **train_kwargs) -> pd.DataFrame:
    rows = []
    total_runs = len(TRAINING_SEEDS) * len(WEIGHT_INIT_SEEDS)
    run_index = 0
    for training_seed in TRAINING_SEEDS:
        for weight_init_name, weight_init_tuple in WEIGHT_INIT_SEEDS.items():
            run_index += 1
            print(
                f"[{model_name} | {threshold_strategy}] run {run_index}/{total_runs}: "
                f"seed={training_seed}, init={weight_init_name}"
            )
            row = train_tf_model(
                model_builder=model_builder,
                model_name=model_name,
                training_seed=training_seed,
                weight_init_name=weight_init_name,
                weight_init_tuple=weight_init_tuple,
                threshold_strategy=threshold_strategy,
                **train_kwargs,
            )
            rows.append(row)
            append_result_row(row, RESULTS_CSV_PATH)
            cleanup_after_run()
    return pd.DataFrame(rows)



## MLP baseline definition

This architecture is intentionally compact so the experiment emphasizes temporal robustness, framework behavior, and threshold estimation rather than architecture tuning.


In [ ]:
def build_mlp_baseline(seed_tuple):
    output_units = 1 if IS_BINARY else NUM_CLASSES
    return keras.Sequential([
        layers.Input(shape=(NUM_FEATURES,)),
        layers.Dense(128, activation="relu", kernel_initializer=initializer_from_seed(weight_seed(seed_tuple, 0))),
        layers.Dense(64, activation="relu", kernel_initializer=initializer_from_seed(weight_seed(seed_tuple, 1))),
        layers.Dense(output_units, kernel_initializer=initializer_from_seed(weight_seed(seed_tuple, 2))),
    ])


## Train and evaluate MLP baseline

This section first runs the baseline with the fixed threshold `0.5`, then repeats the same 30-run seed/init grid using EVT adaptive threshold estimation on the validation-shift scores.


In [ ]:
if RESULTS_CSV_PATH.exists():
    RESULTS_CSV_PATH.unlink()
if AGGREGATED_RESULTS_CSV_PATH.exists():
    AGGREGATED_RESULTS_CSV_PATH.unlink()

def run_threshold_strategy(strategy_name: str) -> pd.DataFrame:
    return run_model_experiment(
        build_mlp_baseline,
        "MLP baseline",
        threshold_strategy=strategy_name,
        epochs=5,
        batch_size=512,
        lr=1e-3,
        use_class_weights=True,
    )

default_results = run_threshold_strategy("default_0.5")
evt_results = run_threshold_strategy("evt_adaptive_threshold")
per_run_results = pd.concat([default_results, evt_results], ignore_index=True)
per_run_results.head()



## Aggregate, compare, plot, and save results

The per-run table preserves every seed/init result for both threshold strategies. The aggregate and comparison tables show whether EVT improves F1, FPR, recall, precision, or ROC-AUC over the fixed `0.5` threshold.


In [ ]:
metric_cols = ["accuracy", "precision", "recall", "f1", "fpr", "roc_auc"]
context_cols = [
    col for col in ["task", "temporal_protocol", "calibration_fraction", "threshold_strategy"]
    if col in per_run_results.columns
]
required_detail_cols = [
    "dataset_name",
    "framework",
    "model_name",
    "task",
    "temporal_protocol",
    "calibration_fraction",
    "threshold_strategy",
    "seed",
    "weight_init",
    "threshold",
    "evt_u",
    "evt_xi_hat",
    "evt_sigma_hat",
    "evt_nu",
    "evt_removed_outliers",
    "evt_min_ks_statistic",
    *metric_cols,
]
required_detail_cols = [col for col in required_detail_cols if col in per_run_results.columns or col in {"seed", "weight_init"}]

detailed_results = per_run_results.copy()
detailed_results["seed"] = detailed_results["training_seed"] if "training_seed" in detailed_results.columns else "default"
detailed_results["weight_init"] = detailed_results["weight_init_name"] if "weight_init_name" in detailed_results.columns else "default"
detailed_results = detailed_results[required_detail_cols]

aggregation_group_cols = ["dataset_name", "framework", "model_name"] + context_cols
aggregation_group_cols = [col for col in aggregation_group_cols if col in detailed_results.columns]
aggregated_results = detailed_results.groupby(aggregation_group_cols, dropna=False)[metric_cols].agg(
    ["mean", "std", "min", "max"]
).round(6)
aggregated_results.columns = [f"{metric}_{stat}" for metric, stat in aggregated_results.columns]
aggregated_results = aggregated_results.reset_index()

before_after_metrics = aggregated_results.pivot_table(
    index=["dataset_name", "framework", "model_name", "task", "temporal_protocol"],
    columns="threshold_strategy",
    values=["accuracy_mean", "precision_mean", "recall_mean", "f1_mean", "fpr_mean", "roc_auc_mean"],
).round(6)
before_after_metrics.columns = [f"{metric}_{strategy}" for metric, strategy in before_after_metrics.columns]
before_after_metrics = before_after_metrics.reset_index()
if {"f1_mean_default_0.5", "f1_mean_evt_adaptive_threshold"}.issubset(before_after_metrics.columns):
    before_after_metrics["delta_f1_evt_minus_default"] = (
        before_after_metrics["f1_mean_evt_adaptive_threshold"] - before_after_metrics["f1_mean_default_0.5"]
    ).round(6)
if {"fpr_mean_default_0.5", "fpr_mean_evt_adaptive_threshold"}.issubset(before_after_metrics.columns):
    before_after_metrics["delta_fpr_evt_minus_default"] = (
        before_after_metrics["fpr_mean_evt_adaptive_threshold"] - before_after_metrics["fpr_mean_default_0.5"]
    ).round(6)

DETAILED_RESULTS_PATH = RESULTS_DIR / f"{NOTEBOOK_FILENAME.replace('.ipynb', '')}_detailed_results.csv"
AGGREGATED_RESULTS_PATH = RESULTS_DIR / f"{NOTEBOOK_FILENAME.replace('.ipynb', '')}_aggregated_results.csv"
COMPARISON_RESULTS_PATH = RESULTS_DIR / f"{NOTEBOOK_FILENAME.replace('.ipynb', '')}_before_after_threshold_comparison.csv"
serializable_per_run_results = per_run_results.copy()
for col in ["confusion_matrix", "classification_report"]:
    if col in serializable_per_run_results.columns:
        serializable_per_run_results[col] = serializable_per_run_results[col].apply(json.dumps)

serializable_per_run_results.to_csv(RESULTS_CSV_PATH, index=False)
detailed_results.to_csv(DETAILED_RESULTS_PATH, index=False)
aggregated_results.to_csv(AGGREGATED_RESULTS_PATH, index=False)
aggregated_results.to_csv(AGGREGATED_RESULTS_CSV_PATH, index=False)
before_after_metrics.to_csv(COMPARISON_RESULTS_PATH, index=False)

print("=== DETAILED RESULTS (60 EVALUATIONS: 30 DEFAULT + 30 EVT) ===")
display(detailed_results)

print("=== AGGREGATED RESULTS BY THRESHOLD STRATEGY ===")
display(aggregated_results)

print("=== BEFORE/AFTER EVT THRESHOLD COMPARISON ===")
display(before_after_metrics)

print("Saved per-run results to:", RESULTS_CSV_PATH)
print("Saved detailed reporting table to:", DETAILED_RESULTS_PATH)
print("Saved aggregated results to:", AGGREGATED_RESULTS_CSV_PATH)
print("Saved before/after threshold comparison to:", COMPARISON_RESULTS_PATH)
print("Saved diagnostic plots under:", PLOTS_DIR)

print("CSV result files saved under:", RESULTS_DIR)
for csv_path in sorted(RESULTS_DIR.glob("*.csv")):
    print(" -", csv_path)




## Final status



In [ ]:
print("=== MLP baseline TensorFlow EVT temporal-shift experiment complete ===")
print("Use the aggregated CSV and before/after comparison CSV for reporting, not a single run.")

